# dcgan-wrapper-netG-netD — faded example 1: Complete the wrapper: register both subnets correctly

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-wrapper-netG-netD`. Running the beacon reports progress on the `Generative: DCGAN netG+netD wrapper` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: DCGAN netG+netD wrapper` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-wrapper-netG-netD`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-wrapper-netG-netD"
DD_SUBTOPIC = "Generative: DCGAN netG+netD wrapper"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The DCGAN wrapper is an `nn.Module` that holds two subnets as attributes and has no `forward`. The critical line is `super().__init__()` — it must run before any submodule assignment, because `nn.Module.__setattr__` needs the `_modules` dict (created inside `__init__`) to register a child module.

## Faded exercise 1

### Faded — register both subnets in the wrapper

Complete `make_wrapper(generator, discriminator)`. The class body assigns `self.netG` and `self.netD`, but the line that makes those assignments legal (and registers them as submodules) is missing. Fill in the one statement that initializes the base `nn.Module` machinery.

**Fill in:** Calls the parent class initializer (`super().__init__()`) so the `_modules` dict exists before submodule assignment.

In [ ]:
from torch import nn

def make_wrapper(generator, discriminator):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            raise NotImplementedError()  # TODO: call the parent nn.Module initializer before assigning submodules
            self.netG = netG
            self.netD = netD
    return DCGAN(generator, discriminator)

gen = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))
disc = nn.Sequential(nn.Linear(5, 8), nn.ReLU(), nn.Linear(8, 1))
model = make_wrapper(gen, disc)


def _test():
    from torch import nn
    g = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))
    d = nn.Sequential(nn.Linear(5, 8), nn.ReLU(), nn.Linear(8, 1))
    m = make_wrapper(g, d)
    assert isinstance(m, nn.Module)
    assert m.netG is g and m.netD is d
    children = list(m.children())
    assert g in children and d in children, 'both subnets must be registered submodules'
    total = sum(p.numel() for p in m.parameters())
    expected = sum(p.numel() for p in g.parameters()) + sum(p.numel() for p in d.parameters())
    assert total == expected
    assert type(m).forward is nn.Module.forward, 'wrapper must not define its own forward'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch import nn

def make_wrapper(generator, discriminator):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    return DCGAN(generator, discriminator)

gen = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))
disc = nn.Sequential(nn.Linear(5, 8), nn.ReLU(), nn.Linear(8, 1))
model = make_wrapper(gen, disc)
```
</details>